In [ ]:
# installation des programmes
!pip install unidecode
!pip install selenium
!pip install ydata_profiling
!pip install rapidfuzz

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
import json
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
import time
from selenium.webdriver import ChromeOptions
from datetime import datetime
from unidecode import unidecode
from ast import literal_eval
from itertools import chain
from ydata_profiling import ProfileReport
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
import unicodedata
import re
import numpy as np
import pandas as pd

In [ ]:
df_Eq = pd.read_excel('/content/Equipement_dataExport (1).xlsx')

In [ ]:
df_Eq_archives = pd.read_excel('/content/Eq_Archives.xlsx')
df_Eq = pd.read_excel('/content/Eq_OPT_FT_SGCB.xlsx')

In [ ]:
df_Eq_archives

In [ ]:
df_Eq.columns

In [ ]:
# @title Titre par défaut
NomSite = ["653 - ECOLE LES PETUNIAS",
            "646 - ECOLE CANDIDE KOCH",
            "125 - MAIRIE DE NOUMEA HOTEL DE VILLE",
            "639 - ECOLE ANTOINETTE CHARBONNEAUX",
            "596 - ECOLE SERGE LAIGLE",
            "645 - ECOLE MARIE HAVET",
            "656 - ECOLE FREDERIC SURLEAU",
            "632 - ECOLE JACQUES TROUILLOT",
            "582 - ECOLE MATHILDE BROQUET",
            "628 - ECOLE MARIE COURTOT",
            "634 - ECOLE CHARLES BICHON",
            "587 - ECOLE LES PENSEES",
            "644 - ECOLE FRANCOIS GRISCELLI",
            "588 - ECOLE AMELIE COSNIER",
            "598 - ECOLE CHRISTINE BOLETTI",
            "651 - ECOLE LES OEILLETS",
            "592 - ECOLE HENRIETTE GERVOLINO",
            "590 - ECOLE LES IRIS",
           "589 - ECOLE TEYSSANDIER DE LAUBAREDE",
           "638 - ECOLE MARGUERITE CARLIER",
           "637 - ECOLE LES CAPUCINES",
           "654 - ECOLE LES ROSES",
           "629 - ECOLE LES HIBISCUS",
           "641 - ECOLE EDMOND DESBROSSE",
           "597 - ECOLE GUSTAVE MOUCHET",
           "648 - ECOLE ADRIENNE LOMONT",
           "633 - ECOLE MICHEL AMIOT",
           "657 - ECOLE DANIEL TALON"
            ]

In [ ]:
#philtre sur le df par nom de site
# df_Eq_1 = df_Eq[df_Eq["🏢 Désignation du site (r_RelationSiteEquipement_1)"].isin(NomSite)].reset_index()

In [ ]:
df_Eq_archives = df_Eq_archives.rename(columns={
    '🏭 Client (r_RelationEntrepriseEquipement_1)' : "Entreprise (r_EquipementsV2Entreprise_1)",
    '🏢 Désignation du site (r_RelationSiteEquipement_1)' : "Désignation du site (r_EquipementsV2DeSignationDuSite_1)",
    "📍 Localisation de l'équipement : BAT / LOCAL (LocalisationDeLEQuipement)" : "Local (Local)",
    "🔢 N° (NEnsemble)" : "Numéro de l'équipement (NumeRoDeLEQuipement)",
    "⚙️ Désignation de l'équipement (DeSignationDeLEQuipement)" : "Désignation de l'équipement (DeSignationDeLEQuipement)",
    "Marque (Marque)" : "Marque de l'équipement (MarqueDeLEQuipement)",
    "🚥 Statut (Statut)" : "Statut de l'équipement (StatutDeLEQuipement)",
    "⚠️ Criticité (Criticite)" : "Criticité (Criticite)",
    "Puissance (Puissance)" : "Puissance de l'équipement (PuissanceDeLEQuipement)",
    "🔢Numéro de série (NDeSeRie)" : "Numéro de série (NumeRoDeSeRie)",
    "Commentaire (Commentaire)" : "Commentaire de maintenance sur l'équipement (CommentaireDeMaintenanceSurLEQuipement)",
    'Mesures (Mesures)' : "Mesures réalisés (MesuresReAliseS)",
    'Dernière date de mise à jour (FR) (DernieReDateDeMiseAJour)' : "Date & heure d'enregistrement de l'équipement (DateHeureDeCreAtion)"
    })

In [ ]:
df_Eq_archives = df_Eq_archives[[
       'Entreprise (r_EquipementsV2Entreprise_1)',
       'Désignation du site (r_EquipementsV2DeSignationDuSite_1)',
       'Local (Local)',
       "Numéro de l'équipement (NumeRoDeLEQuipement)",
       "Désignation de l'équipement (DeSignationDeLEQuipement)",
       "Date & heure d'enregistrement de l'équipement (DateHeureDeCreAtion)",
       "Marque de l'équipement (MarqueDeLEQuipement)",
       "Statut de l'équipement (StatutDeLEQuipement)",
       "Criticité (Criticite)",
       "Puissance de l'équipement (PuissanceDeLEQuipement)",
       "Numéro de série (NumeRoDeSeRie)",
       "Commentaire de maintenance sur l'équipement (CommentaireDeMaintenanceSurLEQuipement)",
       "Mesures réalisés (MesuresReAliseS)",
       ]]

In [ ]:
#Fonction : modifier la designation du site
def extraire_site(valeur):
    if ' - ' in valeur:
        return valeur.split(' - ', 1)[1]
    else:
        return valeur

In [ ]:
df_Eq_archives_copy = df_Eq_archives
df_Eq_archives_copy

In [ ]:
#copie du df
df_Eq_archives_copy = df_Eq_archives
df_Eq_archives_copy['Désignation du site (r_EquipementsV2DeSignationDuSite_1)'] = df_Eq_archives_copy['Désignation du site (r_EquipementsV2DeSignationDuSite_1)'].apply(extraire_site)
df_Eq_archives_copy

In [ ]:
# importer rapidfuzz
import pandas as pd
from rapidfuzz import process, fuzz
from unidecode import unidecode

In [ ]:
# Colonnes utilisées pour la comparaison
cols_compare = [
    'Entreprise (r_EquipementsV2Entreprise_1)',
    'Désignation du site (r_EquipementsV2DeSignationDuSite_1)',
    'Local (Local)',
    "Désignation de l'équipement (DeSignationDeLEQuipement)",
    "Marque de l'équipement (MarqueDeLEQuipement)",
    "Statut de l'équipement (StatutDeLEQuipement)",
    "Criticité (Criticite)"
]

In [ ]:
# Clé de comparaison
df_Eq["compare_key"] = df_Eq[cols_compare].fillna("").astype(str).agg(" | ".join, axis=1)
df_Eq_archives_copy["compare_key"] = df_Eq_archives_copy[cols_compare].fillna("").astype(str).agg(" | ".join, axis=1)

# Liste des clés du dataframe actuel
eq_keys = df_Eq["compare_key"].tolist()

lignes_a_ajouter = []

for _, row in df_Eq_archives_copy.iterrows():

    match = process.extractOne(
        row["compare_key"],
        eq_keys,
        scorer=fuzz.token_sort_ratio
    )

    score = match[1] if match else 0

    # Si aucune correspondance >= 90%
    if score < 90:
        lignes_a_ajouter.append(row.drop("compare_key"))

# DataFrame résultat contenant les lignes de l'archive absentes de df_Eq
df_resultat = pd.DataFrame(lignes_a_ajouter)

print(f"{len(df_resultat)} lignes à ajouter")

In [ ]:
df_resultat

In [ ]:
df_resultat["Entreprise (r_EquipementsV2Entreprise_1)"] = df_resultat["Entreprise (r_EquipementsV2Entreprise_1)"].replace({
    "SGCB ALMA" : "SGCB",
    "SGCB MAGENTA" : "SGCB"
})

In [ ]:
df_resultat["Entreprise (r_EquipementsV2Entreprise_1)"].value_counts()

In [ ]:
df_resultat_copy = df_resultat

In [ ]:
col_site = "Désignation du site (r_EquipementsV2DeSignationDuSite_1)"
col_sous_site = "Désignation du sous site (r_EquipementsV2DeSignationDuBaTiment_1)"

df_resultat_copy[col_sous_site] = np.select(
    [
        df_resultat[col_site].eq("SGCB - FOCH 0 - RDC (13-OA-8257)"),
        df_resultat[col_site].eq("SGCB - FOCH 0 - SALLE INFO (13-OA-8245)")
    ],
    [
        "RDC (13-OA-8257)",
        "SALLE INFO (13-OA-8245)"
    ],
    default= pd.NA
)

In [ ]:
df_resultat_copy

In [ ]:
df_resultat_copy["Désignation du site (r_EquipementsV2DeSignationDuSite_1)"].value_counts()

In [ ]:
df_resultat_copy["Désignation du site (r_EquipementsV2DeSignationDuSite_1)"].replace({
    "SGCB - FOCH 0 - RDC (13-OA-8257)" : "SGCB - FOCH 0",
    "SGCB - FOCH 0 - SALLE INFO (13-OA-8245)" : "SGCB - FOCH 0",
    "SGCB ALMA" : "SGCB - ALMA"
    }, inplace=True)

In [ ]:
df_resultat_copy.to_excel('df_Eq_O_FT_S.xlsx', index=False)